### Four things that are *not* errors

* **Ratios below 1 are expected.** C29 is the whole sector; an assembly plant is part of it.
* **Measurement bases differ** — employer-reported headcount vs Eurostat *persons employed in local units*.
* **Suppression is not random** — and here it is measured in both directions.
* **France IS included, without breaking Rule 1.**

### How France joins the same analysis

Its denominator is **URSSAF, never Eurostat**. URSSAF reports by département (NUTS-3), and départements aggregate exactly into the 22 pre-2016 régions — which *are* French NUTS-2. So France drops onto the same panel, the same GISCO geometry and the same figures.

Two things are carried in the data so the difference is never hidden:

| column | meaning |
|---|---|
| `region_source` | `eurostat` or `urssaf` — France is shaded a **different hue** on the map |
| `numerator_basis` | France uses the plant's **commune** (all auto APE), not a plant headcount |

Why a commune: the reported French plant figure is a single **static** value and cannot form a series. The substitution is close — the commune/plant ratio has **median 0.96** — but it is a commune, and every figure says so.

In [ ]:
# Reload edited .py modules automatically — without this, a kernel that
# imported an older version keeps serving it (AttributeError on new functions).
%load_ext autoreload
%autoreload 2

import os
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import foundation as fd, module01_spatial as m1, module04_macro_micro as m4
import iea_value_chain as iv
# --- archive bootstrap: resolve paths from the project root ---------------
# The original was CONFIG.data_dir = './data', which assumed the notebook ran
# beside a data/ folder. In the archived layout the inputs live in
# 02_data_inputs/, so the path is taken from specular_paths instead and this
# notebook works from wherever it is opened.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent))   # 01_code/
from specular_paths import P
fd.CONFIG.data_dir = str(P.inputs)
fd.CONFIG.out_dir  = str(P.outputs)
print('data:', fd.CONFIG.data_dir)
fd.CONFIG.out_dir  = './outputs'
pd.set_option('display.width', 220)

## 1. Join the two worlds

In [ ]:
prp = m4.plant_region_panel()      # France INCLUDED (URSSAF denominator)
ok  = prp[prp['ratio_flag'] == 'ok']
print('region-years:', len(prp), '| regions:', prp['nuts2'].nunique())
print('region_source:', prp['region_source'].value_counts().to_dict())
print('countries:', sorted(prp['country_iso2'].unique()))
print('median ratio: %.2f' % ok['ratio'].median())
print('\nby source:')
display(ok.groupby('region_source')['ratio'].agg(['median','count']).round(2))
prp.to_csv('outputs/tbl04_plant_region_panel.csv', index=False)

## 2. The coverage ratio

Braunschweig at ~0.86 is the sanity check that the join is right: it hosts **Wolfsburg, Salzgitter and Brunswick**, so the tracked plants really are almost the whole region's automotive sector.

In [ ]:
summ = m4.coverage_ratio_summary(prp)
fig = m4.fig_coverage_ratio(prp, summ, save=str(fd.CONFIG.out('fig04_coverage_ratio.png'))); plt.show()
display(summ.head(12)[['nuts2','geo_label','n_plants','mean_ratio','mean_plant_emp','mean_region_emp','ratio_change']].round(2))
summ.to_csv('outputs/tbl04_coverage_summary.csv', index=False)

## 3. The detector — ratio > 1

Impossible if the data were right, so every hit names a fault. The panel now carries `foundation`'s quality flags, so `implausible_ratios()` reports a **likely cause** rather than leaving you to guess.

Running it both ways separates two very different problems:

| | implausible cases |
|---|---|
| as-is | **9** — all stale plant values |
| `drop_suspect=True` | **1** — a genuine measurement mismatch |

The survivor is **Braunschweig 2023**: VW's three plants (Wolfsburg, Salzgitter, Brunswick) report **84,328** against Eurostat's **82,733** for the entire region — a 2% overshoot, almost certainly because plant headcount includes R&D and administrative staff classified outside NACE 29. That is the measurement-basis caveat, quantified.

In [ ]:
bad = m4.implausible_ratios(prp)
display(bad.round(2))

# with the stale values excluded, only the measurement mismatch remains
prp_clean = m4.plant_region_panel(drop_suspect=True)
print('implausible after dropping suspect rows:', len(m4.implausible_ratios(prp_clean)))
display(m4.implausible_ratios(prp_clean).round(2))

# Ford Genk now ends where it actually closed
clean = fd.employment_panel(drop_post_closure=True, drop_frozen=True)
display(clean[clean['plant_uid']=='BE_FORD_GENK'][['year','employees','value_kind']])

## 4. The residual — the supplier tissue

Regional C29 employment **outside** our tracked plants: bodies, parts, trailers, untracked assemblers. A rising residual share means the supplier ecosystem grew faster than tracked assembly; a falling one means the opposite.

In [ ]:
res = m4.supplier_residual(prp, window=(2012, 2023))
print('balanced regions:', res.attrs['n_regions_balanced'], '| years:', res.attrs['years'])
fig = m4.fig_supplier_residual(res, save=str(fd.CONFIG.out('fig04_supplier_residual.png'))); plt.show()
display(res.pivot(index='year', columns='bloc', values='residual_share').round(3))

## 4b. What the residual is made of

The residual is defined by subtraction — regional automotive employment minus the plants tracked here — which makes it correct but abstract. The IEA facility map gives it a shape. It geolocates 17,156 automotive sites and says what each one is:

| type | sites, worldwide |
|---|---|
| parts supplier | 15,812 |
| tier-1 component supplier | 554 |
| steel plant | 275 |
| car assembly | 248 |
| in-house component | 194 |
| battery cell manufacturing | 39 |
| battery component manufacturing | 34 |

Roughly sixty supplier sites for every assembly plant. The residual is not a statistical artefact — it is that tissue, and it can be counted region by region.

> Points are kept only where they fall inside a NUTS country polygon. A latitude–longitude box wide enough to reach Finland also reaches Oran and Kaliningrad, and those would arrive as European assembly plants the panel is missing when they simply sit outside the reporting area.

In [ ]:
iea = iv.iea_in_europe()
print(f'{len(iea):,} IEA sites inside the NUTS area')
display(iea['type'].value_counts().rename('sites').to_frame())

### 4b.1 How complete is the plant panel?

The assembly points answer a question no internal check can: are there car plants in Europe the panel simply does not contain? Matching is by distance only — the IEA file carries coordinates and a cluster label, never a name — so a match means *a tracked plant lies within X km*, not *these are confirmed to be the same factory*. Because the threshold is a judgement, the whole curve is shown.

In [ ]:
cov = iv.coverage_audit(iea=iea)
fig = iv.fig_coverage_audit(cov, save=str(fd.CONFIG.out('fig04_iea_coverage.png'))); plt.show()
display(cov)

miss = iv.unmatched_assembly(50, iea)
print(f'{len(miss)} IEA assembly sites have no tracked plant within 50 km, by country:')
display(miss['cntr'].value_counts().rename('sites').to_frame())
display(miss.head(12).round(2))

> The gaps are not random. The United Kingdom accounts for ten of them, then Italy, Slovakia and Hungary with four each. The panel was built around the German and French groups, and this quantifies what that leaves out: Volvo at Gothenburg, Nissan at Sunderland, SEAT at Martorell, the Emilian supercar cluster, and several eastern sites. Any European total computed from the panel should be read as a lower bound, and this is the figure that justifies saying so.

### 4b.2 The residual, beside the sites that should explain it

If the residual really is the supplier tissue, regions with more supplier sites should carry more of it.

In [ ]:
dens = iv.supplier_density(level=2, iea=iea)
rvs  = iv.residual_vs_suppliers(prp, year=2023, dens=dens)
fig  = iv.fig_residual_vs_suppliers(rvs, save=str(fd.CONFIG.out('fig04_iea_residual.png'))); plt.show()

print(f"correlation across {len(rvs)} regions: r = {rvs.attrs['pearson_r']:.2f}, "
      f"rank r = {rvs.attrs['spearman_r']:.2f}")
display(rvs.nlargest(10, 'residual')[['nuts2','geo_label','plant_emp','residual','assembly','suppliers']].round(0))

> **r = 0.60, rank correlation 0.64.** The residual sits where the suppliers are, which supports reading it as the tissue around assembly rather than as measurement error.

> Two regions make the point on their own. Île-de-France records 2,100 employees in tracked plants against a residual of 28,600, with 227 supplier sites. Rhône-Alpes records 118 against 19,980, with 250 sites. These are supplier regions with almost no assembly — invisible in a plant panel, and a large part of what the sector actually employs.

### 4b.3 The whole chain on one map

In [ ]:
m = iv.map_value_chain(radius_km=50, year=2023,
                       save=str(fd.CONFIG.out('map04_value_chain.html')))
print('saved ->', os.path.abspath('outputs/map04_value_chain.html'))
m

## 5. Micro–macro divergence

Where plants and their regions moved in **opposite** directions. If plants grew while the region shrank, the surrounding supplier tissue must have contracted — and vice versa. This is the closest these data get to the *upstream transmission* question in the internship brief.

In [ ]:
div = m4.micro_macro_divergence(prp, 2012, 2023)
fig = m4.fig_divergence(div, save=str(fd.CONFIG.out('fig04_divergence.png'))); plt.show()
cols = ['geo_label','country_iso2','plant_emp_growth','region_emp_growth','residual_growth','divergence']
print('PLANTS OUTGREW THEIR REGION'); display(div.head(6)[cols].round(3))
print('REGION OUTGREW ITS PLANTS');   display(div.tail(6)[cols].round(3))
div.to_csv('outputs/tbl04_divergence.csv', index=False)

## 5b. Does a plant follow the region around it?

Sections 2 to 5 compare **levels** — how much of a region's automotive employment sits inside the tracked plants. This compares **movement**.

If a plant and its region rise and fall together, the plant is a fair barometer of its local industry and a regional figure can stand in for it. If they diverge, the region is telling a different story from the factory at its centre, and that divergence is the finding.

The correlation is computed on **year-to-year changes**, not levels. Two series that both drift downward over a decade will correlate almost perfectly in levels whatever they did in between, which says nothing about whether one follows the other.

In [ ]:
pvr = m4.plant_vs_region(window=(2012, 2023))
fig = m4.fig_plant_vs_region(pvr, save=str(fd.CONFIG.out('fig04_plant_vs_region.png'))); plt.show()

c = pvr['corr_changes'].dropna()
print(f'{len(pvr)} plants with at least 8 overlapping years')
print(f'median year-to-year correlation {c.median():+.2f}  |  '
      f'{(c > 0.3).sum()} move with their region, {(c < -0.3).sum()} against it')
display(pvr[['plant_name','country_iso2','geo_label','plant_pct',
             'region_pct','gap_pp','corr_changes']].round(1))

### 5b.1 Six plants beside their regions

In [ ]:
fig = m4.fig_tracking_examples(pvr=pvr,
                               save=str(fd.CONFIG.out('fig04_tracking_examples.png'))); plt.show()

> **The median correlation is +0.29** — plants follow their regions loosely, and about half show no relationship at all. Three cases are worth reading closely.

> **Audi Ingolstadt, +0.95.** The plant essentially *is* the automotive economy of Oberbayern; the two series are the same shape. Where one plant dominates a region, macro and micro measure the same thing.

> **Mercedes Ludwigsfelde, −10% against a region up 131%.** Brandenburg's automotive employment more than doubled while the van plant inside it shrank. The regional figure captures an arrival elsewhere in the region, not the fortunes of the plant.

> **BMW Leipzig, +12% against a region up 95%.** Same pattern, same direction: the plant is a small and shrinking part of a booming regional total.

> The practical conclusion is that a regional series is a poor proxy for a plant unless the plant dominates the region. That is a limit on any employment analysis conducted at NUTS-2 alone, and it is the strongest argument in this study for holding plant-level and regional data side by side rather than choosing between them.

## 6. Suppression is not random

Two mechanisms pull in opposite directions and **both** are visible. Host regions are large enough to report at all (less *not available*) but are dominated by one or two plants (more *confidential*). Reporting this protects the thesis from the charge that it ignored selection built into its own macro source.

In [ ]:
sup = m4.suppression_diagnostic()
display(sup.round(1))
fig = m4.fig_suppression(sup, save=str(fd.CONFIG.out('fig04_suppression.png'))); plt.show()

## 7. France in close-up — commune within département

France already appears in every figure above at NUTS-2. This section zooms further in: URSSAF is **commune-level**, so a plant's own commune can be set against its **département** — the closest to an exact macro–micro link anywhere in the thesis.

> Note on the source fields, which are not what their names suggest: `geo_code` is the **commune INSEE code** (57289 = Hambach) and `region_code` is the **département name** ("Moselle"). The département code is derived from the first two digits of the commune code; the name is kept alongside so any mismatch is visible.

In [ ]:
fr = m4.france_macro_micro()
print('plant-years:', len(fr), '| plants:', fr['plant_uid'].nunique())
display(fr[['city','departement_code','departement','departement_name_src']].drop_duplicates().head(8))
fig = m4.fig_france_macro_micro(fr, save=str(fd.CONFIG.out('fig04_france.png'))); plt.show()
fr.to_csv('outputs/tbl04_france_macro_micro.csv', index=False)

In [ ]:
# which year to map? pick on evidence
yc = m4.year_coverage(prp)
display(yc.round(1))
print('best:', yc.nlargest(3,'score')['year'].tolist(),
      '| worst:', yc.nsmallest(3,'score')['year'].tolist())

## 8. Overlay map

**Reference year = 2023**, chosen from coverage rather than convention. 2019 *looks* like the natural pre-COVID reference and is in fact the **worst year in the series** — 64% of NUTS-2 regions carry a value, against **93% in 2023**. Two effects coincide: 2019–2020 are the weakest years of the 2008–2020 extract, and the 2021+ extract is far better populated.

**Shading** = regional employment: <span style='color:#2C6FB3'>blue</span> Eurostat, <span style='color:#7A5C99'>purple</span> URSSAF (France).

**Bubbles** = tracked plants, sized by employment:

| colour | meaning |
|---|---|
| green | ratio > 0.5 |
| amber | ratio ≤ 0.5 |
| red | ratio > 1 — check the data |
| violet | region **suppressed** by Eurostat |
| grey | outside Eurostat (UK 2021+, TR, RU) |

White regions carry no Eurostat value at all — either withheld as confidential, or genuinely not available.

> Folium needs a **trusted** notebook to render inline.

In [ ]:
# Choose the year from coverage, not habit — see year_coverage() below.
m = m4.map_plants_over_regions(prp, year=2023,
                               save=str(fd.CONFIG.out('map04_plants_over_regions.html')))
print('saved ->', os.path.abspath('outputs/map04_plants_over_regions.html'))
print('If nothing renders below, the notebook is UNTRUSTED (Folium needs inline JS).')
m

---
## What Module 04 establishes

| finding | value |
|---|---|
| median tracked share of regional C29 | **0.39** |
| best-covered region | **Braunschweig 0.86** (Wolfsburg + Salzgitter + Brunswick) |
| regions with a computable ratio | 33 |
| host regions: missing vs other | **11%** vs 28% |
| host regions: confidential vs other | **8.6%** vs 6.6% |
| France: commune ÷ département (median) | 0.27 → 0.25 |

**The headline.** The tracked plants are roughly **two-fifths** of regional automotive employment — so the panel is substantial but not the sector. The complementary three-fifths is the supplier tissue, and its share **fell in both peripheries** while holding in the core: assembly grew faster than its ecosystem where the industry expanded.

**Divergence** identifies the interesting cases directly — Dresden and Thüringen where plants far outgrew their regions, Brandenburg and Madrid where the region moved without the plants.

## Open items

1. **Fix Ford Genk upstream** and audit other plants with long `derived_scraped` runs after a closure.
2. **Ratio > 1 as a standing test** — re-run it after every data update; it is cheap and it catches real faults.
3. **NUTS-1 version** — the crosswalk stores NUTS-2 codes; a NUTS-1 variant would let France join the Eurostat side directly (at the cost of resolution).
4. **Corsica / overseas** département codes (2A/2B, 97x) are not handled by the two-digit rule; irrelevant for the current plants but worth a guard if the panel grows.

**Next — Module 03b** (vehicle mass, once the EEA years land), then the synthesis chapter.